Instalar dependencias


In [ ]:
%pip install -q 'numpy<2.0.0'
%pip install -q esp-ppq==1.3.11
%pip install -q onnx==1.17.0
%pip install -q 'onnxruntime>=1.19.0'
%pip install -q 'torch>=2.4.0'
%pip install -q 'torchvision>=0.19.0'
%pip install -q 'ultralytics==8.4.7'
%pip install -q 'onnxsim>=0.4.36'
%pip install -q roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 75.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but y

🧱 Step 1: Download & Prepare LEGO Minifigures Dataset

In [ ]:
from roboflow import Roboflow
from google.colab import drive
drive.mount('/content/drive')
import os
from dotenv import load_dotenv

# Specify the path to your .env file in Google Drive
dotenv_path = 'direccion de tu .env'

# Load environment variables from the .env file
if os.path.exists(dotenv_path):
    load_dotenv(dotenv_path) # This loads variables into os.environ
    print(f"Loaded .env file from {dotenv_path}")
    # Retrieve the API key from the environment variables
    roboflow_api_key = os.getenv("ROBOFLOW_API_KEY")
else:
    print(f"Warning: .env file not found at {dotenv_path}. Please create one with your ROBOFLOW_API_KEY.")
    # Fallback if .env not found, or if API key isn't in .env
    roboflow_api_key = os.getenv("ROBOFLOW_API_KEY", "YOUR_API_KEY_HERE")

# Ensure roboflow_api_key is a string, even if .env was missing or didn't contain it
if not isinstance(roboflow_api_key, str) or roboflow_api_key == "YOUR_API_KEY_HERE":
    print("Error: Roboflow API key not found or is default. Please set ROBOFLOW_API_KEY in your .env file or hardcode it.")
    # You might want to raise an exception or exit here if the key is critical
    # For now, we'll continue with the placeholder, which will likely fail the Roboflow call


rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace("nombre_de_tu_workspace").project("proyect_ID")
version = project.version(1) #<---- version del proyecto
dataset = version.download("yolo26") #<----Modelo a entrenar


Mounted at /content/drive
Loaded .env file from /content/drive/MyDrive/Colab Notebooks/.env
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Face-recognition-1 in yolo26:: 100%|██████████| 32191/32191 [00:10<00:00, 3197.81it/s] 


Step 2: Fix Absolute Paths in data.yaml

In [ ]:
import yaml
import os

# dataset.location comes from the previous cell (NuJ6TQkbHIt4)
yaml_path = os.path.join(dataset.location, 'data.yaml')

# 1. Read the current yaml
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# 2. Fix the path logic
data['path'] = dataset.location # Use the actual downloaded path
data['train'] = 'train/images'
data['val'] = 'valid/images'
data['test'] = 'test/images'

# 3. Write it back
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("✅ data.yaml paths updated!")
print(f"New 'path' is: {data['path']}")
DRIVE_SAVE_PATH = 'direccion_donde_guardar_el resultado'

✅ data.yaml paths updated!
New 'path' is: /content/Face-recognition-1


Step 3: Fine-Tuning YOLO26n

In [ ]:
from ultralytics import YOLO
import os
import torch

# Disable unwanted loggers
os.environ["COMET_MODE"] = "disabled"
os.environ["WANDB_MODE"] = "disabled"


# Determine the device to use
device = 0 if torch.cuda.is_available() else 'cpu' # Use GPU 0 if available, else CPU
print(f"Using device: {device}")

# Load pre-trained base model
model = YOLO('yolo26n.pt')

# Fine-tune on LEGO dataset
results = model.train(
    data=yaml_path,
    epochs=15,
    imgsz=512,
    batch=24,
    device=device,
    project=os.path.dirname(DRIVE_SAVE_PATH), # Carpeta padre
    name=os.path.basename(DRIVE_SAVE_PATH),    # Nombre de la carpeta del experimento
    plots=True,

    # MuSGD Fine-tuning setup
    optimizer='MuSGD',
    lr0=0.005,
    lrf=0.01,
)

print("✅ Training complete!")

Using device: 0
New https://pypi.org/project/ultralytics/8.4.154 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.7 🚀 Python-3.12.13 torch-2.9.1+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Face-recognition-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Entren

KeyboardInterrupt: 